In [ ]:
from google.colab import drive
import pandas as pd

drive.mount('/content/drive')

file_path = '/content/drive/MyDrive/dataset/CAD Data.xlsx'

df = pd.read_excel(file_path)

df

Mounted at /content/drive


,No,Age,Weight,Length,Sex,BMI,DM,HTN,Current Smoker,EX-Smoker,...,K,Na,WBC,Lymph,Neut,PLT,EF-TTE,Region RWMA,VHD,Cath
0,1,53,90,175,Male,29.387755,0,1,1,0,...,4.7,141,5700,39,52,261,50,0,N,Cad
1,2,67,70,157,Fmale,28.398718,0,1,0,0,...,4.7,156,7700,38,55,165,40,4,N,Cad
2,3,54,54,164,Male,20.077335,0,0,1,0,...,4.7,139,7400,38,60,230,40,2,mild,Cad
3,4,66,67,158,Fmale,26.838648,0,1,0,0,...,4.4,142,13000,18,72,742,55,0,Severe,Normal
4,5,50,87,153,Fmale,37.165193,0,1,0,0,...,4.0,140,9200,55,39,274,50,0,Severe,Normal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
298,299,58,84,168,Male,29.761905,0,0,0,0,...,4.8,146,8500,34,58,251,45,0,N,Cad
299,300,55,64,152,Fmale,27.700831,0,0,0,0,...,4.0,139,11400,16,80,377,40,0,mild,Normal
300,301,48,77,160,Fmale,30.078125,0,1,0,0,...,4.0,140,9000,35,55,279,55,0,N,Normal
301,302,57,90,159,Fmale,35.599858,1,0,0,0,...,3.8,141,3800,48,40,208,55,0,N,Normal


# Menggunakan Library Naive Bayes

In [ ]:
from google.colab import drive
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

# Mount Google Drive
drive.mount('/content/drive')
file_path = '/content/drive/MyDrive/dataset/CAD Data.xlsx'

# Load dataset
df = pd.read_excel(file_path)

# Preprocessing: Encoding categorical variables
label_encoders = {}
for col in df.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

# Splitting dataset
X = df.drop(columns=['Cath'])  # Fitur (hapus kolom target)
y = df['Cath']  # Target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# (a) Menggunakan library Naïve Bayes
nb = GaussianNB()
nb.fit(X_train, y_train)
y_pred = nb.predict(X_test)

# Evaluasi
print("Akurasi dengan sklearn:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Akurasi dengan sklearn: 0.7868852459016393
              precision    recall  f1-score   support

           0       0.92      0.77      0.84        43
           1       0.60      0.83      0.70        18

    accuracy                           0.79        61
   macro avg       0.76      0.80      0.77        61
weighted avg       0.82      0.79      0.79        61



# Tanpa Library Naive Bayes (Masih Menggunakan Numpy)

In [ ]:
from google.colab import drive
import pandas as pd
import numpy as np

# Mount Google Drive
drive.mount('/content/drive')
file_path = '/content/drive/MyDrive/dataset/CAD Data.xlsx'

# Load dataset
df = pd.read_excel(file_path)

# Preprocessing: Encoding categorical variables secara manual
def label_encode(series):
    unique_vals = list(series.unique())
    return series.apply(lambda x: unique_vals.index(x))

for col in df.select_dtypes(include=['object']).columns:
    df[col] = label_encode(df[col])

# Splitting dataset secara manual
def train_test_split_manual(X, y, test_size=0.2):
    indices = np.arange(len(X))
    np.random.shuffle(indices)
    split_idx = int(len(X) * (1 - test_size))
    train_idx, test_idx = indices[:split_idx], indices[split_idx:]
    return X.iloc[train_idx], X.iloc[test_idx], y.iloc[train_idx], y.iloc[test_idx]

X = df.drop(columns=['Cath'])
y = df['Cath']
X_train, X_test, y_train, y_test = train_test_split_manual(X, y, test_size=0.2)

# Implementasi manual Naïve Bayes
class NaiveBayesClassifier:
    def fit(self, X, y):
        self.classes = np.unique(y)
        self.priors = {c: np.mean(y == c) for c in self.classes}
        self.means = {c: X[y == c].mean(axis=0) for c in self.classes}
        self.variances = {c: X[y == c].var(axis=0) for c in self.classes}

    def gaussian_prob(self, x, mean, var):
        return (1 / np.sqrt(2 * np.pi * var)) * np.exp(-((x - mean) ** 2) / (2 * var))

    def predict(self, X):
        predictions = []
        for i in range(len(X)):
            posteriors = {}
            for c in self.classes:
                prior = np.log(self.priors[c])
                likelihood = np.sum(np.log(self.gaussian_prob(X.iloc[i], self.means[c], self.variances[c])))
                posteriors[c] = prior + likelihood
            predictions.append(max(posteriors, key=posteriors.get))
        return np.array(predictions)

# Training dan uji model manual
nb_manual = NaiveBayesClassifier()
nb_manual.fit(X_train, y_train)
y_pred_manual = nb_manual.predict(X_test)

# Evaluasi manual
def accuracy_score_manual(y_true, y_pred):
    return np.mean(y_true == y_pred)

def classification_report_manual(y_true, y_pred):
    classes = np.unique(y_true)
    report = ""
    precisions, recalls, f1_scores, supports = [], [], [], []
    for c in classes:
        tp = np.sum((y_true == c) & (y_pred == c))
        fp = np.sum((y_true != c) & (y_pred == c))
        fn = np.sum((y_true == c) & (y_pred != c))
        support = np.sum(y_true == c)
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        report += f"Class {c}: Precision={precision:.2f}, Recall={recall:.2f}, F1-Score={f1:.2f}, Support={support}\n"
        precisions.append(precision)
        recalls.append(recall)
        f1_scores.append(f1)
        supports.append(support)

    macro_avg = (np.mean(precisions), np.mean(recalls), np.mean(f1_scores))
    weighted_avg = (
        np.average(precisions, weights=supports),
        np.average(recalls, weights=supports),
        np.average(f1_scores, weights=supports)
    )

    report += f"\nMacro Avg: Precision={macro_avg[0]:.2f}, Recall={macro_avg[1]:.2f}, F1-Score={macro_avg[2]:.2f}\n"
    report += f"Weighted Avg: Precision={weighted_avg[0]:.2f}, Recall={weighted_avg[1]:.2f}, F1-Score={weighted_avg[2]:.2f}\n"
    return report

print("Akurasi dengan implementasi manual:", accuracy_score_manual(y_test, y_pred_manual))
print(classification_report_manual(y_test, y_pred_manual))

Mounted at /content/drive
Akurasi dengan implementasi manual: 0.819672131147541
Class 0: Precision=0.84, Recall=0.90, F1-Score=0.87, Support=40
Class 1: Precision=0.78, Recall=0.67, F1-Score=0.72, Support=21

Macro Avg: Precision=0.81, Recall=0.78, F1-Score=0.79
Weighted Avg: Precision=0.82, Recall=0.82, F1-Score=0.82



# TANPA LIBRARY NAIVE BAYES & TANPA NUMPY

In [ ]:
from google.colab import drive
import pandas as pd

# Mount Google Drive
drive.mount('/content/drive')
file_path = '/content/drive/MyDrive/dataset/CAD Data.xlsx'

# Load dataset
df = pd.read_excel(file_path)

# Encoding manual untuk mengubah nilai kategori ke angka
def label_encode(series):
    unique_vals = list(series.unique())
    return series.apply(lambda x: unique_vals.index(x))

for col in df.select_dtypes(include=['object']).columns:
    df[col] = label_encode(df[col])

# Shuffle list tanpa library random
def shuffle_list(lst):
    for i in range(len(lst) - 1, 0, -1):
        j = i % (i + 1)  # Pseudo-random swap
        lst[i], lst[j] = lst[j], lst[i]

# Fungsi split dataset manual
def train_test_split_manual(X, y, test_size=0.2):
    indices = list(range(len(X)))
    shuffle_list(indices)
    split_idx = int(len(X) * (1 - test_size))
    train_idx, test_idx = indices[:split_idx], indices[split_idx:]
    return X.iloc[train_idx], X.iloc[test_idx], y.iloc[train_idx], y.iloc[test_idx]

X = df.drop(columns=['Cath'])
y = df['Cath']
X_train, X_test, y_train, y_test = train_test_split_manual(X, y, test_size=0.2)

# Implementasi Naïve Bayes Manual
class NaiveBayesClassifier:
    def fit(self, X, y):
        self.classes = list(set(y))
        self.priors = {c: sum(y == c) / len(y) for c in self.classes}
        self.means = {c: X[y == c].mean(axis=0) for c in self.classes}
        self.variances = {c: X[y == c].var(axis=0) + 1e-6 for c in self.classes}  # Tambahkan epsilon agar tidak 0

    def gaussian_prob(self, x, mean, var):
        return (1 / ((2 * 3.14159 * var) ** 0.5)) * (2.71828 ** (-((x - mean) ** 2) / (2 * var)))

    def predict(self, X):
        predictions = []
        for i in range(len(X)):
            posteriors = {}
            for c in self.classes:
                prior = self.priors[c]
                likelihood = 1
                for j in range(len(X.iloc[i])):
                    likelihood *= self.gaussian_prob(X.iloc[i, j], self.means[c].iloc[j], self.variances[c].iloc[j])
                posteriors[c] = prior * likelihood
            predictions.append(max(posteriors, key=posteriors.get))
        return predictions

# Training dan testing
nb_manual = NaiveBayesClassifier()
nb_manual.fit(X_train, y_train)
y_pred_manual = nb_manual.predict(X_test)

# Evaluasi
def accuracy_score_manual(y_true, y_pred):
    return sum(y_true.iloc[i] == y_pred[i] for i in range(len(y_true))) / len(y_true)

def classification_report_manual(y_true, y_pred):
    classes = list(set(y_true))
    report = "Classification Report:\n"
    macro_precision, macro_recall, macro_f1 = 0, 0, 0
    weighted_precision, weighted_recall, weighted_f1 = 0, 0, 0

    for c in classes:
        tp = sum((y_true.iloc[i] == c) and (y_pred[i] == c) for i in range(len(y_true)))
        fp = sum((y_true.iloc[i] != c) and (y_pred[i] == c) for i in range(len(y_true)))
        fn = sum((y_true.iloc[i] == c) and (y_pred[i] != c) for i in range(len(y_true)))
        support = sum(y_true == c)

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

        report += f"Class {c}: Precision={precision:.2f}, Recall={recall:.2f}, F1-Score={f1:.2f}, Support={support}\n"

        macro_precision += precision
        macro_recall += recall
        macro_f1 += f1
        weighted_precision += precision * support
        weighted_recall += recall * support
        weighted_f1 += f1 * support

    n_classes = len(classes)
    total_support = len(y_true)

    report += f"\nMacro Avg: Precision={macro_precision/n_classes:.2f}, Recall={macro_recall/n_classes:.2f}, F1-Score={macro_f1/n_classes:.2f}"
    report += f"\nWeighted Avg: Precision={weighted_precision/total_support:.2f}, Recall={weighted_recall/total_support:.2f}, F1-Score={weighted_f1/total_support:.2f}"

    return report

print("Akurasi dengan implementasi manual:", accuracy_score_manual(y_test, y_pred_manual))
print(classification_report_manual(y_test, y_pred_manual))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Akurasi dengan implementasi manual: 0.5573770491803278
Classification Report:
Class 0: Precision=0.78, Recall=0.22, F1-Score=0.34, Support=32
Class 1: Precision=0.52, Recall=0.93, F1-Score=0.67, Support=29

Macro Avg: Precision=0.65, Recall=0.57, F1-Score=0.50
Weighted Avg: Precision=0.65, Recall=0.56, F1-Score=0.50


# KESIMPULAN

Berdasarkan hasil pengujian tiga versi implementasi Naïve Bayes, performa model bervariasi tergantung pada penggunaan library. Model yang menggunakan pandas dan numpy memiliki akurasi tertinggi, yaitu 81.97%, dengan keseimbangan yang baik antara precision dan recall. Pada model ini, Class 0 memiliki precision sebesar 0.84 dan recall 0.90, sedangkan Class 1 memiliki precision 0.78 dan recall 0.67, menunjukkan bahwa model mampu mengenali kedua kelas dengan cukup baik.

Sementara itu, model yang menggunakan sklearn memiliki akurasi 78.69%, sedikit lebih rendah dibandingkan model manual dengan numpy. Model ini memiliki precision yang sangat tinggi untuk Class 0 (0.92) tetapi recall-nya hanya 0.77, menunjukkan bahwa model lebih cenderung mengklasifikasikan Class 0 dengan lebih akurat. Sebaliknya, Class 1 memiliki recall yang lebih tinggi (0.83), tetapi precision yang lebih rendah (0.60), yang berarti model lebih sering mengklasifikasikan data sebagai Class 1 meskipun belum tentu benar.

Di sisi lain, model yang hanya menggunakan pandas tanpa numpy memiliki akurasi terendah, yaitu 55.74%, yang menunjukkan bahwa model ini kurang optimal dalam melakukan klasifikasi. Model ini memiliki recall yang sangat rendah untuk Class 0 (0.22), yang berarti banyak sampel dari Class 0 diklasifikasikan dengan salah. Sebaliknya, Class 1 memiliki recall sangat tinggi (0.93) tetapi precision yang rendah (0.52), yang mengindikasikan banyak false positives. Ketidakseimbangan ini menyebabkan model kurang efektif untuk kedua kelas.

Dari ketiga model yang diuji, model dengan pandas dan numpy merupakan yang terbaik, karena memiliki akurasi tertinggi, distribusi precision dan recall yang lebih seimbang, serta F1-score yang lebih baik dibandingkan model lainnya. Model dengan sklearn masih menjadi alternatif yang cukup baik, tetapi kurang optimal dalam mengenali Class 1. Sementara itu, model yang hanya menggunakan pandas tidak direkomendasikan karena memiliki performa yang jauh lebih rendah dibandingkan dua model lainnya.